# Make finished SIF and uSIF tables on your own corpus
fse performs inference. Learn weights and uncentered principal components here, then fold them into each word row.
The equations follow [SIF](https://openreview.net/forum?id=SyK00v5xx) and [uSIF](https://aclanthology.org/W18-3012/).
The reference for the implementation choices is [fse 1.1.0 on 1.x](https://github.com/oborchers/Fast_Sentence_Embeddings/tree/1.x/src/fse/models):
SIF uses `a/(a+p)`. uSIF estimates `a` from vocabulary size and average sentence length, then uses `a/(a/2+p)`.
uSIF removes components with weights proportional to their squared singular values, not the squares of those weights.
The decomposition is exact and uncentered, using a Gram matrix and SciPy. Folding is offline and need not preserve bits.

This experiment fits on STS-B **train texts only**, without scores. It evaluates test Pearson correlation times 100.
It uses corpus frequencies, not the English wordfreq estimates of the historical README experiment.
GloVe uses lowercase regex tokens. POTION uses its original tokenizer without added special tokens and drops unknown IDs.
POTION is already distilled and weighted. Additional SIF/uSIF is an experiment, not a recommended default.
All downloads and generated models stay in a temporary directory. An existing exact-revision Hub cache is read only.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import os
import platform
from datetime import date

scratch = TemporaryDirectory(prefix="fse-recipe-")
work = Path(scratch.name)
# Third-party caches also belong to this temporary directory.
for name in ("HF_HOME", "XDG_CACHE_HOME", "NUMBA_CACHE_DIR", "MPLCONFIGDIR"):
    os.environ[name] = str(work / name.lower())
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np
import fse
print(date.today().isoformat(), platform.platform(), platform.machine())

In [ ]:
import csv
import re
from collections import Counter
from scipy.linalg import eigh
from scipy.stats import pearsonr
from huggingface_hub import hf_hub_download
from gensim.models import KeyedVectors
from safetensors.numpy import load_file
from tokenizers import Tokenizer

repository = Path.cwd().parent if Path.cwd().name == "recipes" else Path.cwd()

def split(name):
    with (repository / "evaluation" / f"sts-{name}.csv").open(encoding="utf-8") as stream:
        rows = list(csv.reader(stream, delimiter="\t", quoting=csv.QUOTE_NONE))
    return [r[5] for r in rows], [r[6] for r in rows], np.array([float(r[4]) for r in rows])
train_a, train_b, _ = split("train")
test_a, test_b, gold = split("test")

def source_file(repo, revision, filename):
    cached = Path.home() / ".cache/huggingface/hub" / ("models--" + repo.replace("/", "--")) / "snapshots" / revision / filename
    if cached.is_file():
        return cached
    return Path(hf_hub_download(repo, filename, revision=revision, cache_dir=work / "hub"))

glove_repo, glove_revision = "fse/glove-wiki-gigaword-300", "4e5ad402664236b85be45a0d107b95259ba8a909"
# Both files must be beside each other for gensim's native saved format.
for filename in ("glove-wiki-gigaword-300.model", "glove-wiki-gigaword-300.model.vectors.npy"):
    source_file(glove_repo, glove_revision, filename)
glove = KeyedVectors.load(str(source_file(glove_repo, glove_revision, "glove-wiki-gigaword-300.model")), mmap="r")
potion_repo, potion_revision = "minishlab/potion-base-8M", "bf8b056651a2c21b8d2565580b8569da283cab23"
potion_rows = load_file(source_file(potion_repo, potion_revision, "model.safetensors"))["embeddings"]
potion_tokenizer = Tokenizer.from_file(str(source_file(potion_repo, potion_revision, "tokenizer.json")))
potion_words = [potion_tokenizer.id_to_token(i) for i in range(len(potion_rows))]
unknown = potion_tokenizer.token_to_id("[UNK]")

def glove_tokens(text):
    return re.findall(r"[a-z0-9]+(?:'[a-z]+)?", text.lower())
def potion_tokens(text):
    return [potion_words[i] for i in potion_tokenizer.encode(text, add_special_tokens=False).ids if i != unknown]
print({"train_pairs": len(train_a), "test_pairs": len(test_a), "glove_revision": glove_revision, "potion_revision": potion_revision})

In [ ]:
def probabilities(words, corpus):
    counts = Counter(token for tokens in corpus for token in tokens)
    values = np.array([counts[word] for word in words], dtype=np.float64)
    if values.sum() == 0:
        raise ValueError("corpus has no vocabulary overlap")
    return values / values.sum()

def word_weights(p, method, length):
    if method == "mean":
        return np.ones(len(p), dtype=np.float32)
    if method == "sif":
        a = 1e-3
        return (a / (a + p)).astype(np.float32)
    if method != "usif":
        raise ValueError(method)
    v = len(p)
    threshold = 1 - (1 - 1/v) ** length
    alpha = np.count_nonzero(p > threshold) / v
    a = (1-alpha) / (alpha * (v/2) + np.finfo(np.float32).eps)
    weights = a / (a/2 + p)
    if not np.isfinite(weights).all():
        raise ValueError("uSIF weights are not finite")
    return weights.astype(np.float32)

def means(table, words, corpus):
    lookup = {word: i for i, word in enumerate(words)}
    rows = np.zeros((len(corpus), table.shape[1]), dtype=np.float32)
    for i, tokens in enumerate(corpus):
        ids = [lookup[t] for t in tokens if t in lookup]
        if ids:
            # Like 1.1.0, the offline mean counts unknown tokens in its denominator.
            rows[i] = table[ids].sum(axis=0, dtype=np.float32) / len(tokens)
    return rows

def components(rows, k, weighted):
    gram = np.zeros((rows.shape[1], rows.shape[1]), dtype=np.float64)
    for start in range(0, len(rows), 16384):
        block = rows[start:start+16384]
        gram += block.T @ block  # float32 block product, float64 accumulation
    values, basis = eigh(gram, driver="evd")
    values = np.maximum(values[-k:][::-1], 0)
    basis = basis[:, -k:][:, ::-1].T.astype(np.float32)
    factors = (values / values.sum()).astype(np.float32) if weighted else np.ones(k, dtype=np.float32)
    return basis, factors

def remove(rows, basis, factors):
    return rows - ((rows @ basis.T) * factors) @ basis

def unit(rows):
    norms = np.linalg.norm(rows, axis=1, keepdims=True)
    return np.divide(rows, norms, out=np.zeros_like(rows), where=norms != 0)

def score(rows):
    a, b = rows[:len(gold)], rows[len(gold):]
    return float(pearsonr(gold, np.einsum("ij,ij->i", a, b)).statistic * 100)

In [ ]:
results = []
for name, table, words_, tokenize in (
    ("GloVe-300", glove.vectors, glove.index_to_key, glove_tokens),
    ("POTION-8M", potion_rows, potion_words, potion_tokens),
):
    train_tokens = [tokenize(t) for t in train_a + train_b]
    test_tokens = [tokenize(t) for t in test_a + test_b]
    p = probabilities(words_, train_tokens)
    length = max(1, int(np.mean([len(t) for t in train_tokens])))
    for method in ("mean", "sif", "usif"):
        weights = word_weights(p, method, length)
        weighted = np.asarray(table * weights[:, None], dtype=np.float32)
        if method == "mean":
            basis, factors = np.empty((0, table.shape[1]), dtype=np.float32), np.empty(0, dtype=np.float32)
        else:
            fitted = means(weighted, words_, train_tokens)
            basis, factors = components(fitted, 10 if method == "sif" else 5, method == "usif")
        # Fold in bounded blocks, so no vocabulary-sized intermediate projection is needed.
        folded = np.empty_like(weighted)
        for start in range(0, len(weighted), 16384):
            folded[start:start+16384] = remove(weighted[start:start+16384], basis, factors)
        finished = fse.Encoder.from_arrays(folded, words=words_, tokenizer="external", normalize=True)
        # External tokenization is an explicit part of this recipe. Save it alongside the model in a real application.
        path = work / f"{name}-{method}.safetensors"
        finished.save(path)
        finished = fse.Encoder.load(path, mmap=True)
        encoded = finished.encode(test_tokens, workers=1)
        reference = unit(remove(means(weighted, words_, test_tokens), basis, factors))
        error = float(np.max(np.abs(encoded-reference)))
        np.testing.assert_allclose(encoded, reference, atol=3e-5, rtol=3e-5)
        results.append({"table": name, "method": method, "pearson_x100": score(encoded), "fold_max_abs_error": error, "mean_train_length": length})
        del finished, weighted, folded
for row in results:
    print(row)

## Read the measured comparison
The rows above compare each method with the mean on the same table, split and tokenizer.
A gain here is evidence for this STS-B setup only. It does not establish gains on a production corpus.
The historical README uses test-fitted components, different preprocessing and English wordfreq frequencies.
Its GloVe-300 values (mean 44.46, SIF-10 71.43, uSIF 68.82) are not direct targets for this train-fitted experiment.
The next section reproduces that historical setup separately to check the migration.

## Share only after approval
The model is a finished table. No recipe metadata or learned components enter the fse artifact.
Keep the external tokenizer with the application. Do not use this artifact with raw strings.
The following is **text only**, not an executed cell. Review licensing, model card and tokenizer distribution before publishing:

```python
finished.push_to_hub("you/my-finished-model", license="pddl")
```
PDDL applies to the GloVe source. POTION uses MIT. Review the license of your corpus separately.

In [ ]:
# Historical diagnostic only: fit on the test texts as the 1.x notebook did.
# Do not use test-fitted components when reporting held-out quality.
import nltk
from nltk import word_tokenize
from wordfreq import get_frequency_dict
nltk.download("punkt_tab", download_dir=str(work / "nltk"), quiet=True, raise_on_error=True)
nltk.data.path.insert(0, str(work / "nltk"))

def historical_tokens(text):
    result = []
    for token in word_tokenize(text):
        if re.match(r".*[A-Za-z0-9].*", token):
            token = token.lower().strip("';.:()").strip('"')
            result.extend(re.split("-", "not" if token == "n't" else token))
    return result
historical = [historical_tokens(t) for t in test_a + test_b]
frequency = get_frequency_dict("en", wordlist="best")
counts = np.array([int(frequency.get(w, 1e-8) * (2**31-1)) for w in glove.index_to_key], dtype=np.int64)
p = counts.astype(np.float32)
p /= counts.sum()
legacy_results = []
for method, published in (("mean", 44.46), ("sif", 71.43), ("usif", 68.82)):
    weighted = glove.vectors * word_weights(p, method, 11)[:, None]
    rows = means(weighted, glove.index_to_key, historical)
    if method != "mean":
        basis, factors = components(rows, 10 if method == "sif" else 5, method == "usif")
        reference = unit(remove(rows, basis, factors))
        folded = np.empty_like(weighted)
        for start in range(0, len(weighted), 16384):
            folded[start:start+16384] = remove(weighted[start:start+16384], basis, factors)
    else:
        reference, folded = unit(rows), weighted
    model = fse.Encoder.from_arrays(folded, words=glove.index_to_key, tokenizer="external", normalize=True)
    migrated = model.encode(historical, workers=1)
    np.testing.assert_allclose(migrated, reference, atol=3e-5, rtol=3e-5)
    measured = score(migrated)
    legacy_results.append({"method": method, "recipe_pearson_x100": measured, "README_1_1_0": published, "delta_points": measured-published})
    del model, weighted, folded
for row in legacy_results:
    print(row)

In [ ]:
scratch.cleanup()